# Laboratorio 4 — Fundamentos de Aprendizaje por Refuerzo y Gymnasium

**CC3092 · Deep Learning y Sistemas Inteligentes**
Autor: Esteban Cárcamo (`ecarcamo`)

Este notebook contiene los tres bloques del laboratorio:

1. **Investigación** de los fundamentos del aprendizaje por refuerzo (RL).
2. **Investigación** de la librería **Gymnasium**.
3. **Módulo de prueba**: un agente aleatorio en dos entornos (CartPole-v1 y
   FrozenLake-v1) y una política simple *no aprendida* para CartPole-v1, con las
   gráficas de recompensa por episodio y la comparación de desempeño.

> Cada afirmación de las secciones 1 y 2 se acompaña de al menos una fuente. Las
> referencias completas están al final de cada sección.

## 1. Fundamentos del aprendizaje por refuerzo

El aprendizaje por refuerzo (*Reinforcement Learning*, RL) estudia cómo un
**agente** aprende a tomar decisiones interactuando con un **entorno** para
maximizar una señal de recompensa acumulada. A continuación se investigan sus
conceptos fundamentales.

### 1.1 ¿Qué es RL y en qué se diferencia del aprendizaje supervisado y no supervisado?

El **aprendizaje por refuerzo** es un paradigma en el que un agente aprende *por
ensayo y error*: en cada instante observa el estado del entorno, ejecuta una
acción y recibe una **recompensa** escalar (posiblemente retrasada). Su objetivo
es descubrir, sin que nadie le indique la acción correcta, la estrategia que
maximiza la recompensa acumulada a lo largo del tiempo.

Diferencias clave con los otros paradigmas:

| | Aprendizaje **supervisado** | Aprendizaje **no supervisado** | Aprendizaje por **refuerzo** |
|---|---|---|---|
| Datos | Pares (entrada, etiqueta correcta) | Datos sin etiqueta | Experiencia (estado, acción, recompensa) generada al interactuar |
| Señal | *Instructiva*: cuál era la respuesta correcta | Ninguna; busca estructura | *Evaluativa*: qué tan buena fue la acción |
| Objetivo | Generalizar un mapeo entrada→salida | Descubrir patrones/estructura | Maximizar la recompensa acumulada |

Además, en RL **las acciones afectan los estados y las recompensas futuras** (los
datos no son i.i.d., hay dependencia temporal), la recompensa puede llegar con
retraso —lo que crea el problema de **asignación de crédito**— y aparece el
dilema **exploración vs. explotación**, ausente en los otros paradigmas.

*Fuente:* Sutton & Barto (2018), cap. 1.

### 1.2 Componentes de un problema de RL

- **Agente** (*agent*): el que decide y aprende.
- **Entorno** (*environment*): todo lo externo al agente con lo que este interactúa.
- **Estado** (*state*) $s$: representación de la situación del entorno en un instante.
- **Acción** (*action*) $a$: la decisión que toma el agente y que influye en el entorno.
- **Recompensa** (*reward*) $r$: señal escalar de retroalimentación que **define el objetivo**.
- **Política** (*policy*) $\pi$: la estrategia del agente; un mapeo de estados a
  acciones, $\pi(a\mid s)$ (determinista o estocástica).

El ciclo de interacción: en el paso $t$ el agente observa $s_t$, elige $a_t$ según
$\pi$, y el entorno responde con la recompensa $r_{t+1}$ y el nuevo estado
$s_{t+1}$. Este bucle *agente ⇄ entorno* es exactamente la API que implementa
Gymnasium (`reset`/`step`).

*Fuente:* Sutton & Barto (2018), cap. 3; Gymnasium — *Basic Usage*.

### 1.3 Proceso de Decisión de Markov (MDP)

Un **MDP** es el marco matemático formal de un problema de RL. Se define por la
tupla $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

- $\mathcal{S}$: conjunto de **estados**.
- $\mathcal{A}$: conjunto de **acciones**.
- $P$: **función de transición**, $P(s' \mid s, a)$ = probabilidad de pasar a $s'$
  dado que en $s$ se tomó $a$ (la *dinámica* del entorno).
- $R$: **función de recompensa**, $R(s, a, s')$ = recompensa esperada de esa transición.
- $\gamma \in [0, 1]$: **factor de descuento**, que pondera cuánto valen las
  recompensas futuras frente a las inmediatas.

Se apoya en la **propiedad de Markov**: el futuro depende únicamente del estado y
la acción actuales, no de toda la historia. El objetivo es hallar la política que
maximiza el **retorno esperado** $G_t = \sum_{k \ge 0} \gamma^{k} r_{t+k+1}$.

*Fuente:* Sutton & Barto (2018), cap. 3; Bellman (1957).

### 1.4 Funciones de valor $V(s)$ y $Q(s,a)$

- **Función de valor de estado** $V^{\pi}(s) = \mathbb{E}_{\pi}[\,G_t \mid S_t = s\,]$:
  el retorno esperado si se parte de $s$ y luego se sigue la política $\pi$.
  Responde: *¿qué tan bueno es estar en este estado?*
- **Función de valor acción-estado** $Q^{\pi}(s,a) = \mathbb{E}_{\pi}[\,G_t \mid S_t = s, A_t = a\,]$:
  el retorno esperado si se parte de $s$, se toma la acción $a$ y luego se sigue $\pi$.
  Responde: *¿qué tan buena es esta acción en este estado?*

**Relación entre ambas:** $V^{\pi}(s) = \sum_a \pi(a\mid s)\, Q^{\pi}(s,a)$.

**Relación con la política óptima:** la política óptima $\pi^{*}$ es *greedy*
respecto de los valores óptimos, $\pi^{*}(s) = \arg\max_a Q^{*}(s,a)$, y
$V^{*}(s) = \max_a Q^{*}(s,a)$. Conocer $Q^{*}$ basta para actuar de forma óptima
**sin necesitar el modelo** $P$ del entorno; por eso $Q$ es la base de métodos de
control como Q-Learning.

*Fuente:* Sutton & Barto (2018), cap. 3.

### 1.5 Ecuación de Bellman

**Intuición:** el valor de un estado se descompone de forma recursiva en la
recompensa inmediata más el valor descontado del estado al que se llega. En una
frase: *«el valor de donde estoy = lo que gano ahora + el valor (descontado) de a
donde voy»*.

Ecuación de Bellman **de expectación** (para una política $\pi$):

$$V^{\pi}(s) = \sum_a \pi(a\mid s) \sum_{s'} P(s'\mid s,a)\,\big[\,R(s,a,s') + \gamma\, V^{\pi}(s')\,\big].$$

Ecuación de Bellman **de optimalidad**:

$$Q^{*}(s,a) = \sum_{s'} P(s'\mid s,a)\,\Big[\,R(s,a,s') + \gamma \max_{a'} Q^{*}(s',a')\,\Big].$$

**Rol:** convierte el cálculo de las funciones de valor en un problema de *punto
fijo*. Es el fundamento de la programación dinámica (*value/policy iteration*) y
de los métodos de diferencias temporales: Q-Learning, por ejemplo, usa el lado
derecho de la ecuación de optimalidad como *objetivo* (*target*) de cada
actualización.

*Fuente:* Sutton & Barto (2018), cap. 3–4.

### 1.6 Dilema exploración vs. explotación

El agente enfrenta un compromiso permanente: **explotar** (elegir la mejor acción
conocida para maximizar la recompensa ahora) frente a **explorar** (probar
acciones inciertas para descubrir opciones potencialmente mejores). Explotar
siempre puede dejarlo atrapado en un óptimo local; explorar siempre desperdicia
recompensa. Dos estrategias habituales:

- **$\varepsilon$-greedy:** con probabilidad $1-\varepsilon$ toma la acción
  *greedy* ($\arg\max_a Q(s,a)$) y con probabilidad $\varepsilon$ toma una acción
  aleatoria uniforme. Es simple y suele usarse con $\varepsilon$ que **decae** con
  el tiempo (más exploración al inicio, más explotación al final).
- **Softmax / Boltzmann:** elige cada acción con probabilidad proporcional a
  $\exp(Q(s,a)/\tau)$. La **temperatura** $\tau$ regula la aleatoriedad: $\tau$
  alto → casi uniforme (mucha exploración); $\tau$ bajo → casi *greedy*. A
  diferencia de $\varepsilon$-greedy, explora *proporcionalmente* a lo prometedora
  que es cada acción.

*Fuente:* Sutton & Barto (2018), cap. 2.

### 1.7 Tareas episódicas/continuas, on/off-policy y model-based/model-free

- **Episódicas vs. continuas.** Las tareas *episódicas* tienen estados terminales
  y se dividen en episodios independientes (p.ej. una partida que termina); las
  *continuas* no tienen fin natural, y el descuento $\gamma < 1$ garantiza que el
  retorno sea finito.
- **On-policy vs. off-policy.** Un método *on-policy* mejora la **misma** política
  que genera los datos (p.ej. SARSA). Un método *off-policy* aprende una política
  *objetivo* distinta de la política de *comportamiento* que explora (p.ej.
  Q-Learning aprende la política greedy mientras explora con $\varepsilon$-greedy).
- **Model-based vs. model-free.** Los métodos *model-based* aprenden o usan un
  modelo de la dinámica $P$ y la recompensa $R$ para **planificar** (p.ej. Dyna).
  Los *model-free* aprenden valores o política directamente de la experiencia, sin
  modelo explícito (p.ej. Q-Learning, SARSA).

*Fuente:* Sutton & Barto (2018), cap. 3, 5, 6 y 8.

### 1.8 Q-Learning

Q-Learning es un algoritmo **model-free**, **off-policy** y de **diferencias
temporales (TD)** que aprende directamente la función de valor óptima $Q^{*}$. Tras
observar la transición $(s, a, r, s')$ actualiza:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{objetivo TD}} - Q(s,a)\big].$$

El término entre corchetes es el **error TD**: la diferencia entre el objetivo
*bootstrapeado* ($r + \gamma \max_{a'} Q(s',a')$) y la estimación actual.

- **$\alpha$ (tasa de aprendizaje, $0 < \alpha \le 1$):** cuánto se corrige la
  estimación con cada muestra. Un $\alpha$ alto aprende rápido pero de forma
  inestable/ruidosa; un $\alpha$ bajo es estable pero lento.
- **$\gamma$ (factor de descuento, $0 \le \gamma \le 1$):** cuánto pesan las
  recompensas futuras. $\gamma \to 0$ hace al agente *miope* (solo la recompensa
  inmediata); $\gamma \to 1$ lo hace *previsor* (valora el largo plazo).

Es *off-policy* porque en el objetivo usa $\max_{a'}$ (la acción greedy) aunque la
acción realmente ejecutada haya sido otra (exploratoria): así converge a $Q^{*}$
con independencia de la política de exploración.

*Fuente:* Watkins & Dayan (1992); Sutton & Barto (2018), cap. 6.

### Referencias de la sección 1

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction*
  (2.ª ed.). MIT Press. <http://incompleteideas.net/book/the-book-2nd.html>
- Watkins, C. J. C. H., & Dayan, P. (1992). *Q-learning*. Machine Learning, 8(3–4),
  279–292.
- Bellman, R. (1957). *Dynamic Programming*. Princeton University Press.
- Farama Foundation. *Gymnasium Documentation*. <https://gymnasium.farama.org/>